# Source to Bronze Process Overview

This notebook ingests ERP source data from Azure ADLS Gen2 raw storage into Bronze Delta tables registered in Unity Catalog.

## End-to-end process

1. **Initialize logging**
   * Creates a production-style logger to capture ingestion progress, warnings, and failures.

2. **Capture run date context**
   * Derives the current date, month, and year.
   * Uses these values to build the source path for daily-partitioned raw files.

3. **Configure storage and target schema**
   * Defines the Azure storage account and ABFSS paths for Bronze, Silver, and Gold containers.
   * Sets the raw input location and Bronze Delta output location.
   * Creates the target Unity Catalog schema if it does not already exist.

4. **Define source entities to ingest**
   * Lists the ERP source datasets that will be processed in this run:
     * `items`
     * `itemCategories`
     * `salesCreditMemoLines`
     * `salesCreditMemos`
     * `salesInvoiceLines`
     * `salesInvoices`
     * `CustomerCard`
     * `SalesPersons`

5. **Load each source into Bronze**
   * For each source table, the notebook:
     * Reads raw JSON files from the daily folder structure in ADLS.
     * Checks whether data exists before writing.
     * Enriches records with ingestion metadata such as:
       * `ingestion_time`
       * `source_metadata_file`
     * Writes the data in Delta format to the Bronze storage location.
     * Registers the Delta location as a Unity Catalog table using the naming pattern `bz_<table_name>`.

6. **Handle schema evolution and failures**
   * Uses schema merge during writes so new upstream fields can be absorbed.
   * Logs failures with detailed error information without stopping the overall process summary.

7. **Execute ingestion sequentially**
   * Processes all configured source tables one by one.
   * Stores the result of each table load in a run summary list.

8. **Publish run summary**
   * Logs the final status for every table as `SUCCESS`, `SKIPPED`, or `FAILED`.

## Outcome

At the end of the notebook run, raw ERP JSON files for the current processing date are landed as Bronze Delta tables in `erp_lakehouse.bronze`, with ingestion lineage metadata added for traceability.

In [0]:
import logging
import sys
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql.functions import current_timestamp, col,lit

# ==========================================
# 1. SETUP PRODUCTION LOGGING
# ==========================================
logger = logging.getLogger("ERP_Bronze_Ingestion")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(asctime)s - [%(levelname)s] - %(message)s'))
    logger.addHandler(handler)



In [0]:
from datetime import datetime

current_date = datetime.now().date()
current_month = datetime.now().month
current_year = datetime.now().year

In [0]:
print(current_date)
print(current_month)
print(current_year)

In [0]:
# ==========================================
# 2. AZURE ADLS GEN2 STORAGE CONFIGURATION
# ==========================================
# Update these values to match your Azure infrastructure
STORAGE_ACCOUNT = "bbmanufacturingprod"  # Your ADLS Gen2 Storage Account Name
RAW="bronze"
SILVER="silver"
GOLD="gold"

# Production URL structure using secure ABFSS protocol
RAW_ADLS_PATH = f"abfss://{RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_ADLS_PATH = f"abfss://{SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
GOLD_ADLS_PATH = f"abfss://{GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

RAW_DIR = f"{RAW_ADLS_PATH}/raw"
BRONZE_DIR = f"{RAW_ADLS_PATH}/delta"

CATALOG_NAME='erp_lakehouse'

# Target Database/Schema
BRONZE_SCHEMA = "bronze"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")

# List of source systems tables to ingest
FILES_TO_PROCESS = [
    "items",
    "itemCategories",
    "salesCreditMemoLines",
    "salesCreditMemos",
    "salesInvoiceLines",
    "salesInvoices",
    "CustomerCard",
    "SalesPersons"
]



In [0]:
# ==========================================
# 3. ATOMIC INGESTION FUNCTION
# ==========================================
def load_to_bronze(table_name: str) -> str:
    """Reads raw JSON from ADLS Gen2 and writes to external Delta Tables."""
    source_path = f"{RAW_DIR}/{table_name}/{current_year}/{current_month}/{current_date}/*.json"
    target_table = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bz_{table_name}"
    target_path = f"{BRONZE_DIR}/bz_{table_name}"
    
    try:
        logger.info(f"📥 Starting Ingestion for: {table_name}")
        
        # 1. Read JSON with optimized options
        # multiLine=True handles pretty-printed JSON payloads from APIs gracefully
        df = (
            spark.read
            .format("json")
            .load(source_path)
        )
        
        # Guard Check: Avoid writing empty transactions
        if df.isEmpty():
            logger.warning(f"⚠️ {table_name} source data is empty. Skipping database write.")
            return f"{table_name}: SKIPPED (No Data)"

        # 2. Enrich with essential lineage tracking metadata
        processed_df = (
            df.withColumn("ingestion_time", current_timestamp())
              .withColumn("source_metadata_file", col("_metadata.file_path"))  # Tracks exactly which file this record came from
        )
        
        # 3. Write to external Delta table pointing directly to ADLS Gen2
        (
            processed_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")  # Gracefully handle dynamic upstream API changes
            .save(target_path)    # Explicitly stores data in your ADLS directory
        )

        # Step 4: Register to Unity Catalog
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {target_table}
            USING DELTA
            LOCATION '{target_path}'
        """)
        
        logger.info(f"✅ Successful Ingestion: {target_table} saved to storage.")
        return f"{table_name}: SUCCESS"

    except Exception as e:
        logger.error(f"💥 Failed Ingestion for {table_name}. Error: {str(e)}", exc_info=True)
        return f"{table_name}: FAILED"


In [0]:
# ==========================================
# 4. SEQUENTIAL EXECUTION ENGINE
# ==========================================
# Instead of parallel processing, we process tables sequentially.
logger.info("🚀 Initializing sequential orchestration layer...")

results = []
for table_name in FILES_TO_PROCESS:
    result = load_to_bronze(table_name)
    results.append(result)

# Final execution sanity check summary
logger.info("📊 --- INGESTION RUN SUMMARY ---")
for result in results:
    logger.info(f"  {result}")